# Module 8 · LLM Evaluation Frameworks & Traceability

**Architectural Layer:** LLM Evaluation  
**Toolchain:** DeepEval · RAGAS · Langfuse · Weights & Biases  
**Objective:** Evaluate LLM outputs for faithfulness, relevance, and safety; assess RAG pipeline quality; and trace multi-step agent executions.

---

## 🧠 Why LLM Evaluation Is Fundamentally Different

### The Core Problem

Traditional ML: "Is the prediction correct?" → accuracy = 94.3% ✅  
LLM: "Is this paragraph good?" → ???

There's no single number that captures LLM quality. A response can be:
- ✅ Grammatically perfect but ❌ factually wrong (hallucination)
- ✅ Factually correct but ❌ not answering the question (irrelevant)
- ✅ Safe and relevant but ❌ uses information NOT in the context (unfaithful)

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Vibes are not metrics. Eye-balling 10 prompts to see if 'they look better' is amateur engineering. Seniors establish automated **Evaluation-Driven Development (EDD)** frameworks (DeepEval, Ragas) to statistically prove a prompt update improved the baseline before merging code.

**Mental Model:** Treat LLMs like high-variance junior employees. You wouldn't push their code straight to production without a test suite. LLM-as-a-Judge Eval acts as your automated Senior Reviewer grading the responses on a strict 1-10 rubric.

---


## 📑 Table of Contents

* [1 · Evaluation-Driven Development (EDD)](#1--evaluation-driven-development-edd)
* [2 · Building Golden Datasets & Human Evals](#2--building-golden-datasets--human-evals)
* [3 · DeepEval: CI/CD Unit Tests for LLMs](#3--deepeval-cicd-unit-tests-for-llms)
* [4 · RAGAS: Evaluating RAG Pipelines](#4--ragas-evaluating-rag-pipelines)
* [5 · Langfuse: Tracing Agentic Workflows](#5--langfuse-tracing-agentic-workflows)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


---
## 1 · Evaluation-Driven Development (EDD)

Test-Driven Development (TDD) relies on deterministic assertions (`assert 2+2 == 4`). In Generative AI, we use **Evaluation-Driven Development (EDD)**.

**The Workflow:**
1. **Define Criteria:** Compile a 'Golden Dataset' of 50-100 inputs and perfect expected outputs.
2. **Write Evaluators:** Create LLM-as-a-judge scorers for relevance and hallucination.
3. **Run Baseline:** Score your current prompt/model.
4. **Experiment:** Change the prompt, add retrieved context, or swap to a faster model.
5. **Re-Eval:** Did the average score go up? Did you regress on edge cases? If pass, merge to CI/CD.


---
## 2 · Building Golden Datasets & Human Evals

You cannot rely entirely on LLM-as-a-judge. Who judges the judge? A high-quality evaluation pipeline starts with **Human Annotators**.

### Inter-Annotator Agreement (Cohen's Kappa)
If Human A rates a response 5/5, and Human B rates it 2/5, your rubric is broken. We statically measure agreement using Cohen's Kappa before ever letting an LLM judge the data.


In [ ]:
# Cell 1 - Measuring Inter-Annotator Agreement (Cohen's Kappa)
from sklearn.metrics import cohen_kappa_score
import numpy as np

# Two Subject-Matter Experts (SMEs) graded 10 LLM responses for 'Helpfulness' (1=Bad, 0=Good)
SME_A_scores = [1, 0, 1, 1, 0, 1, 0, 0, 1, 0]
SME_B_scores = [1, 0, 0, 1, 0, 1, 0, 1, 1, 0]

kappa = cohen_kappa_score(SME_A_scores, SME_B_scores)

print("🧑‍🔬 Human Eval Reliability Check")
print("-"*40)
print(f"Cohen's Kappa Score: {kappa:.2f}")

if kappa < 0.2:
    print("❌ Slight agreement. Your rubric is ambiguous! The judge will fail.")
elif kappa < 0.6:
    print("⚠️ Moderate agreement. Need clearer definitions of 'Helpful'.")
else:
    print("✅ High agreement (>0.6). The rubric is golden. We can now prompt an LLM-as-a-judge to mimic this!")


---
## 3 · DeepEval: CI/CD Unit Tests for LLMs

Once your rubric is mathematically sound, you automate it using tools like DeepEval. This hooks directly into Github Actions (as seen in `Module 7 · CICD Automation`).


In [ ]:
# Cell 2 — DeepEval Simulation
test_cases = [
    {"id": "TC-001", "input": "What was Q3 revenue?", "context": "Q3 revenue was $52M.", "faithfulness": 1.0, "relevance": 0.95},
    {"id": "TC-002", "input": "Who is CEO?", "context": "John founded it.", "faithfulness": 0.3, "relevance": 0.7}
]

FAITHFULNESS_THRESHOLD = 0.7
RELEVANCE_THRESHOLD = 0.7

print("🧪 DeepEval CI/CD Output:")
for tc in test_cases:
    passed = tc['faithfulness'] >= FAITHFULNESS_THRESHOLD and tc['relevance'] >= RELEVANCE_THRESHOLD
    print(f"{tc['id']} → Faithfulness: {tc['faithfulness']} | ✅ PASS" if passed else f"{tc['id']} → ❌ FAIL: Hallucination detected. Pipeline BLOCKED.")


---
## 4 · RAGAS: Evaluating RAG Pipelines

RAGAS evaluates the ENTIRE pipeline natively. 
- **Context Precision:** Are relevant docs at the TOP of results? (Fix: Reranker)
- **Context Recall:** Did we find ALL relevant docs? (Fix: Better Embeddings)
- **Faithfulness:** Is the answer based ONLY on context? (Fix: System Prompt)


In [ ]:
# Cell 3 — Evaluate a mock RAG pipeline
import pandas as pd

rag_dataset = [
    {"query": "Max context for GPT-4?", "context_precision": 1.0, "context_recall": 1.0, "faithfulness": 1.0},
    {"query": "How does LoRA work?", "context_precision": 0.0, "context_recall": 0.0, "faithfulness": 0.0},
]
df = pd.DataFrame(rag_dataset)
print("📊 RAG Pipeline Evaluation:")
print(df.to_string(index=False))


---
## 5 · Langfuse: Tracing Agentic Workflows
When a test fails, you need to know why. Langfuse explicitly wraps your code to trace multi-step spans natively.


In [ ]:
# Cell 4 — Production tracing with @observe (code reference)
production_code = '''
from langfuse.decorators import observe
import litellm

@observe()  # ← This single decorator captures EVERYTHING in the call tree
def agent_pipeline(query: str) -> str:
    category = classify_query(query)
    context = retrieve_context(category)
    return generate_response(query, context)
'''
print("📄 Production Langfuse Integration:")
print(production_code)


---
## ✅ Knowledge Check

### Q1: EDD CI/CD Blockers
In an EDD framework, why is it critical to separate Context Precision failures from Faithfulness failures?
<details><summary>Answer</summary>
Because they require entirely different engineering fixes. A Context Precision failure means the retrieval architecture (Vector DB, Rerankers) is broken. A Faithfulness failure means the LLM generation process (Prompting, Model choice) is broken. If you have bad retrieval, your faithfulness will naturally suffer, meaning you should fix retrieval first.
</details>

### Q2: Cohen's Kappa
Why do we calculate Cohen's Kappa between human subject-matter experts before configuring an LLM-as-a-judge?
<details><summary>Answer</summary>
If human experts can't reliably agree on what constitutes a 'good' or 'faithful' answer (a low Kappa score), the rubric is fundamentally flawed. An LLM cannot mathematically enforce a definition that humans themselves find ambiguous.
</details>

---

## 🔨 Practical Practice

### Exercise 1: LLM-as-a-judge System Prompt
Write a strict system prompt for GPT-4o that forces it to act as an evaluator for 'Toxicity'. It must output a JSON object containing a `score` (0.0 to 1.0) and a `reasoning` string.

### Exercise 2: Implementing Kappa
Write a script that takes a CSV of 100 responses graded by Human A, 100 graded by Human B, and 100 graded by `gpt-4o`. Calculate the human-human agreement, and the human-AI agreement.


---
## 🎯 Summary & Key Takeaways

| Framework Concept | Tool | Mechanical Impact |
|-------------------|------|-------------------|
| Golden Datasets | Human Annotators | Defines unquestionable Ground Truth |
| Inter-Annotator Agreement | Cohen's Kappa | Mathematically proves the rubric is unambiguous |
| LLM Unit Tests | DeepEval | Plugs into CI/CD to block hallucinating code PRs |
| Pipeline Assessment | RAGAS | Isolates retrieval bugs from generation bugs |
| Workflow Tracing | Langfuse | Logs nested agentic sub-calls and latency overhead |

**Next →** `Module 8 · AI Gateways Resilience` — Protecting production APIs with rate-limiting and fallbacks.
